In [1]:
import os
import pickle
import random
import networkx as nx

# Explicit imports for all 5 requested graph types
from networkx.algorithms import bipartite
from networkx import (
    barabasi_albert_graph,
    erdos_renyi_graph,
    random_regular_graph,  
    watts_strogatz_graph
)


# **Generate Graph Dataset-Watts-Strogatz**

In [1]:

# Create directories for train and test data.
os.makedirs("train", exist_ok=True)
os.makedirs("test", exist_ok=True)

def generate_graphs(num_graphs, min_n, max_n, folder_name):
    for i in range(num_graphs):
        g_type = 'watts-strogatz'

        # Determine the total number of nodes for this graph.
        n = random.randint(min_n, max_n)

        # Find all valid values for 'k' (each node is joined with its k nearest neighbors).
        # k must be an even integer greater than or equal to 2.
        # k must be strictly less than n.
        valid_ks = [k for k in range(2, n) if k % 2 == 0]
        
        # Randomly select a valid uniform degree from the available options.
        k = random.choice(valid_ks)

        # Set 'p' (probability of rewiring each edge).
        # Using a random float between 0.1 and 0.9 ensures a mix of small-world properties.
        p = random.uniform(0.1, 0.9)

        # Generate the Watts-Strogatz small-world graph.
        G = watts_strogatz_graph(n, k, p)

        # Extract metadata and features like adjacency matrix and edge list.
        adj_matrix = nx.to_numpy_array(G)
        edge_list = list(G.edges())
        degrees = dict(G.degree())

        data = {
            'id': i,
            'type': g_type,
            'num_vertices': n,
            'num_edges': G.number_of_edges(),
            'adj_matrix': adj_matrix,
            'edge_list': edge_list,
            'degrees': degrees
        }

        # Save each dictionary using pickle into the target directory.
        filename = os.path.join(folder_name, f"graph_{i:04d}.pkl")
        with open(filename, 'wb') as f:
            pickle.dump(data, f)

# Generate 1000 training graphs with max vertices 15.
# Using 6 as lower bound to maintain consistency with size constraints.
generate_graphs(1000, 6, 15, "train")

# Generate 200 test graphs with vertices between 20 and 150.
generate_graphs(200, 20, 150, "test")

In [ ]:
!unzip /content/train.zip -d /content/
!unzip /content/test.zip -d /content/

Archive:  /content/train.zip
   creating: /content/train/
  inflating: /content/train/ba_00000.pkl  
  inflating: /content/train/ba_00001.pkl  
  inflating: /content/train/ba_00002.pkl  
  inflating: /content/train/ba_00003.pkl  
  inflating: /content/train/ba_00004.pkl  
  inflating: /content/train/ba_00005.pkl  
  inflating: /content/train/ba_00006.pkl  
  inflating: /content/train/ba_00007.pkl  
  inflating: /content/train/ba_00008.pkl  
  inflating: /content/train/ba_00009.pkl  
  inflating: /content/train/ba_00010.pkl  
  inflating: /content/train/ba_00011.pkl  
  inflating: /content/train/ba_00012.pkl  
  inflating: /content/train/ba_00013.pkl  
  inflating: /content/train/ba_00014.pkl  
  inflating: /content/train/ba_00015.pkl  
  inflating: /content/train/ba_00016.pkl  
  inflating: /content/train/ba_00017.pkl  
  inflating: /content/train/ba_00018.pkl  
  inflating: /content/train/ba_00019.pkl  
  inflating: /content/train/ba_00020.pkl  
  inflating: /content/train/ba_00021.pk

# **Simple GNN**

In [2]:
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
import pickle

# ==========================================
# 1. DATASET LOADING
# ==========================================
def load_dataset(folder_name):
    dataset = []
    if not os.path.exists(folder_name):
        print(f"Warning: folder '{folder_name}' not found.")
        return dataset
    for filename in sorted(os.listdir(folder_name)):
        if filename.endswith('.pkl'):
            with open(os.path.join(folder_name, filename), 'rb') as f:
                data = pickle.load(f)
                G = nx.Graph()
                G.add_nodes_from(range(data['num_vertices']))
                G.add_edges_from(data['edge_list'])
                dataset.append({
                    'id'        : data['id'],
                    'type'      : data['type'],
                    'graph'     : G,
                    'adj_matrix': torch.tensor(data['adj_matrix'], dtype=torch.float32),
                    'num_nodes' : data['num_vertices']
                })
    return dataset

# ==========================================
# 2. NORMALISED ADJACENCY (FIX #2)
# ==========================================
def normalized_adj(adj_matrix: torch.Tensor) -> torch.Tensor:

    A = adj_matrix + torch.eye(adj_matrix.size(0))
    D = A.sum(dim=1)
    D_inv_sqrt = torch.diag(1.0 / torch.sqrt(D.clamp(min=1e-8)))
    return D_inv_sqrt @ A @ D_inv_sqrt

# ==========================================
# 3. GROUND TRUTH — GREEDY + REDUCTION (FIX #1)
# ==========================================
def get_ground_truth_vertex_cover(graph):
    # Phase 1: greedy high-degree cover
    G_copy = graph.copy()
    cover_set = set()
    while G_copy.number_of_edges() > 0:
        node = max(G_copy.degree(), key=lambda x: x[1])[0]
        cover_set.add(node)
        G_copy.remove_node(node)

    # Phase 2: remove redundant nodes
    improved = True
    while improved:
        improved = False
        for node in list(cover_set):
            cover_set.remove(node)
            still_valid = all(
                u in cover_set or v in cover_set
                for u, v in graph.edges()
            )
            if still_valid:
                improved = True          # node was redundant, keep removed
            else:
                cover_set.add(node)      # node is needed, restore it

    label = torch.zeros(graph.number_of_nodes(), dtype=torch.float32)
    for node in cover_set:
        label[node] = 1.0
    return label

# ==========================================
# 4. BA-AWARE FEATURE EXTRACTION
# ==========================================
def extract_ba_features(graph):
    n = graph.number_of_nodes()
    degrees = dict(graph.degree())
    max_deg = max(degrees.values()) if max(degrees.values()) > 0 else 1

    clustering     = nx.clustering(graph)
    triangles      = nx.triangles(graph)
    max_tri        = max(triangles.values()) if max(triangles.values()) > 0 else 1
    deg_centrality = nx.degree_centrality(graph)

    avg_neighbor_deg = {}
    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        avg_neighbor_deg[node] = (
            np.mean([degrees[nb] for nb in neighbors]) / max_deg
            if neighbors else 0.0
        )

    features = []
    for node in range(n):
        features.append([
            degrees[node] / max_deg,
            clustering[node],
            triangles[node] / max_tri,
            avg_neighbor_deg[node],
            deg_centrality[node],
        ])

    return torch.tensor(features, dtype=torch.float32)

# ==========================================
# 5. GNN MODEL
# ==========================================
class BAGraphGNN(nn.Module):

    def __init__(self, input_dim, hidden_dim, K_iterations):
        super(BAGraphGNN, self).__init__()
        self.K = K_iterations

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.1),
        )

        self.W1 = nn.Linear(hidden_dim, hidden_dim, bias=False)  # 1-hop
        self.W2 = nn.Linear(hidden_dim, hidden_dim, bias=False)  # 2-hop
        self.U  = nn.Linear(hidden_dim, hidden_dim, bias=False)  # self

        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.activation = nn.LeakyReLU(0.1)
        self.dropout    = nn.Dropout(p=0.3)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, norm_adj):
        h = self.input_proj(x)

        for _ in range(self.K):
            m1 = torch.matmul(norm_adj, self.W1(h))                         # 1-hop
            m2 = torch.matmul(norm_adj, torch.matmul(norm_adj, self.W2(h))) # 2-hop

            h_self = self.U(h)
            h_new  = self.activation(h_self + m1 + 0.5 * m2)
            h_new  = self.layer_norm(h_new)
            h      = self.dropout(h_new)

        return self.sigmoid(self.classifier(h))

# ==========================================
# 6. VALIDITY CHECK
# ==========================================
def is_valid_vertex_cover(graph, selected_nodes):
    selected = set(selected_nodes)
    return all(u in selected or v in selected for u, v in graph.edges())

# ==========================================
# 7. POST-PROCESSING — GREEDY COVER REPAIR
# ==========================================
def greedy_cover_from_probs(probs, edges, num_nodes):
    """
    Add nodes greedily (highest GNN probability first) until
    all edges are covered. Guarantees a valid vertex cover
    regardless of the raw GNN output quality.
    """
    sorted_nodes   = sorted(range(len(probs)), key=lambda k: probs[k], reverse=True)
    selected       = []
    remaining      = list(edges)

    for node in sorted_nodes:
        selected.append(node)
        remaining = [e for e in remaining
                     if e[0] not in selected and e[1] not in selected]
        if not remaining:
            break

    return selected

# ==========================================
# 8. MAIN EXECUTION
# ==========================================
INPUT_DIM  = 5
HIDDEN_DIM = 16
K_HOPS     = 4

print("Loading datasets...")
train_data = load_dataset('train')
test_data  = load_dataset('test')

# ---- Build Training Pairs ----
print("Extracting features and labels for watts-strogatz graphs...")
training_pairs = []
for data in train_data:
    if data['type'] != 'watts-strogatz':
        continue

    x        = extract_ba_features(data['graph'])
    norm_adj = normalized_adj(data['adj_matrix'])          # FIX #2 applied
    y        = get_ground_truth_vertex_cover(data['graph']).unsqueeze(1)  # FIX #1 applied
    training_pairs.append((x, norm_adj, y))

print(f"Found {len(training_pairs)} watts-strogatz training graphs.")

# ---- Train ----
model     = BAGraphGNN(INPUT_DIM, HIDDEN_DIM, K_HOPS)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
criterion = nn.BCELoss()

epochs = 1000
print(f"\nStarting training for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, norm_adj, y in training_pairs:
        optimizer.zero_grad()

        pred      = model(x, norm_adj)
        loss_bce  = criterion(pred, y)

        # Sparsity penalty: push model toward smaller covers
        # λ=0.3 is appropriate for BA; hubs make covers naturally compact
        loss = loss_bce + 0.05 * pred.mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    if (epoch + 1) % 20 == 0:
        avg = total_loss / max(1, len(training_pairs))
        lr  = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg:.4f} | LR: {lr:.6f}")

# ---- Test ----
print("\n" + "=" * 45)
print("TESTING ON UNSEEN watts-strogatz GRAPHS")
print("=" * 45)

model.eval()

valid_count   = 0
better_count  = 0
equal_count   = 0
total         = 0
approx_ratios = []

for data in test_data:
    if data['type'] != 'watts-strogatz':
        continue

    x        = extract_ba_features(data['graph'])
    norm_adj = normalized_adj(data['adj_matrix'])

    with torch.no_grad():
        probs = model(x, norm_adj)

    node_probs     = probs.view(-1).tolist()
    selected_nodes = greedy_cover_from_probs(
        node_probs, list(data['graph'].edges()), data['num_nodes']
    )

    predicted_size = len(selected_nodes)
    is_valid       = is_valid_vertex_cover(data['graph'], selected_nodes)

    # Baseline: NetworkX 2-approximation
    approx_vc   = nx.algorithms.approximation.min_weighted_vertex_cover(data['graph'])
    approx_size = len(approx_vc)

    total += 1
    if is_valid:
        valid_count += 1
        ratio = predicted_size / approx_size
        approx_ratios.append(ratio)

    # FIX #3: clearer metric — beat and match baseline separated
    beats_baseline_strict = is_valid and (predicted_size < approx_size)
    matches_baseline      = is_valid and (predicted_size == approx_size)

    if beats_baseline_strict:
        better_count += 1
    if matches_baseline:
        equal_count += 1

    verdict = "beats baseline" if beats_baseline_strict else (
              "matches baseline" if matches_baseline else (
              "valid but larger" if is_valid else "INVALID COVER"))

    print(f"Graph ID : {data['id']} | Nodes: {data['num_nodes']}")
    print(f" > Valid Cover   : {is_valid}")
    print(f" > Predicted Size: {predicted_size}")
    print(f" > Baseline Size : {approx_size}  ({verdict})")
    print(f" > Selected Nodes: {selected_nodes}")
    print("-" * 40)

# ---- Final Report ----
print("\n" + "=" * 45)
print("FINAL RESULTS")
print("=" * 45)
validity     = (valid_count  / total * 100) if total > 0 else 0
beat_pct     = (better_count / total * 100) if total > 0 else 0
match_pct    = (equal_count  / total * 100) if total > 0 else 0
avg_ratio    = np.mean(approx_ratios) if approx_ratios else float('inf')

print(f"Total Graphs Tested   : {total}")
print(f"Valid Covers          : {valid_count}/{total}  ({validity:.1f}%)")
print(f"Beats Baseline        : {better_count}/{total}  ({beat_pct:.1f}%)")
print(f"Matches Baseline      : {equal_count}/{total}  ({match_pct:.1f}%)")
print(f"Avg Approx Ratio      : {avg_ratio:.3f}  (vs baseline; <1.0 means smaller cover)")
print(f"Note: Baseline is NetworkX 2-approximation, NOT true optimum.")
print("=" * 45)

Loading datasets...
Extracting features and labels for watts-strogatz graphs...
Found 1000 watts-strogatz training graphs.

Starting training for 1000 epochs...
Epoch 20/1000 | Loss: 0.4568 | LR: 0.005000
Epoch 40/1000 | Loss: 0.4540 | LR: 0.005000
Epoch 60/1000 | Loss: 0.4512 | LR: 0.005000
Epoch 80/1000 | Loss: 0.4518 | LR: 0.005000
Epoch 100/1000 | Loss: 0.4483 | LR: 0.002500
Epoch 120/1000 | Loss: 0.4421 | LR: 0.002500
Epoch 140/1000 | Loss: 0.4398 | LR: 0.002500
Epoch 160/1000 | Loss: 0.4427 | LR: 0.002500
Epoch 180/1000 | Loss: 0.4408 | LR: 0.002500
Epoch 200/1000 | Loss: 0.4463 | LR: 0.001250
Epoch 220/1000 | Loss: 0.4391 | LR: 0.001250
Epoch 240/1000 | Loss: 0.4377 | LR: 0.001250
Epoch 260/1000 | Loss: 0.4414 | LR: 0.001250
Epoch 280/1000 | Loss: 0.4433 | LR: 0.001250
Epoch 300/1000 | Loss: 0.4395 | LR: 0.000625
Epoch 320/1000 | Loss: 0.4352 | LR: 0.000625
Epoch 340/1000 | Loss: 0.4388 | LR: 0.000625
Epoch 360/1000 | Loss: 0.4378 | LR: 0.000625
Epoch 380/1000 | Loss: 0.4409 | L

# **Edge Based GNN**

In [3]:
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
import pickle

# ==========================================
# 1. DATASET LOADING
# ==========================================
def load_dataset(folder_name):
    dataset = []
    if not os.path.exists(folder_name):
        print(f"Warning: folder '{folder_name}' not found.")
        return dataset
    for filename in sorted(os.listdir(folder_name)):
        if filename.endswith('.pkl'):
            with open(os.path.join(folder_name, filename), 'rb') as f:
                data = pickle.load(f)
                G = nx.Graph()
                G.add_nodes_from(range(data['num_vertices']))
                G.add_edges_from(data['edge_list'])
                dataset.append({
                    'id'        : data['id'],
                    'type'      : data['type'],
                    'graph'     : G,
                    'adj_matrix': torch.tensor(data['adj_matrix'], dtype=torch.float32),
                    'num_nodes' : data['num_vertices']
                })
    return dataset

# ==========================================
# 2. EDGE STRUCTURE EXTRACTION (LINE GRAPH)
# ==========================================
def get_edge_structures(graph):
    edges = list(graph.edges())
    N = graph.number_of_nodes()
    M = len(edges)

    B = torch.zeros((N, M), dtype=torch.float32)
    for j, (u, v) in enumerate(edges):
        B[u, j] = 1.0
        B[v, j] = 1.0

    if M == 0:
        return B, torch.zeros((0, 0), dtype=torch.float32)

    # Line graph adjacency A_L = B^T * B - 2I
    A_L = B.t() @ B - 2.0 * torch.eye(M)
    A_L = torch.clamp(A_L, min=0.0)

    # Normalize A_L
    A_L_self = A_L + torch.eye(M)
    D_L = A_L_self.sum(dim=1)
    D_L_inv_sqrt = torch.diag(1.0 / torch.sqrt(D_L.clamp(min=1e-8)))
    norm_A_L = D_L_inv_sqrt @ A_L_self @ D_L_inv_sqrt

    return B, norm_A_L

# ==========================================
# 3. GROUND TRUTH — GREEDY + REDUCTION
# ==========================================
def get_ground_truth_vertex_cover(graph):
    # Phase 1: greedy high-degree cover
    G_copy = graph.copy()
    cover_set = set()
    while G_copy.number_of_edges() > 0:
        node = max(G_copy.degree(), key=lambda x: x[1])[0]
        cover_set.add(node)
        G_copy.remove_node(node)

    # Phase 2: remove redundant nodes
    improved = True
    while improved:
        improved = False
        for node in list(cover_set):
            cover_set.remove(node)
            still_valid = all(
                u in cover_set or v in cover_set
                for u, v in graph.edges()
            )
            if still_valid:
                improved = True          # node was redundant, keep removed
            else:
                cover_set.add(node)      # node is needed, restore it

    label = torch.zeros(graph.number_of_nodes(), dtype=torch.float32)
    for node in cover_set:
        label[node] = 1.0
    return label

# ==========================================
# 4. BA-AWARE FEATURE EXTRACTION
# ==========================================
def extract_ba_features(graph):
    n = graph.number_of_nodes()
    degrees = dict(graph.degree())
    max_deg = max(degrees.values()) if max(degrees.values()) > 0 else 1

    clustering     = nx.clustering(graph)
    triangles      = nx.triangles(graph)
    max_tri        = max(triangles.values()) if max(triangles.values()) > 0 else 1
    deg_centrality = nx.degree_centrality(graph)

    avg_neighbor_deg = {}
    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        avg_neighbor_deg[node] = (
            np.mean([degrees[nb] for nb in neighbors]) / max_deg
            if neighbors else 0.0
        )

    features = []
    for node in range(n):
        features.append([
            degrees[node] / max_deg,
            clustering[node],
            triangles[node] / max_tri,
            avg_neighbor_deg[node],
            deg_centrality[node],
        ])

    return torch.tensor(features, dtype=torch.float32)

# ==========================================
# 5. EDGE-BASED GNN MODEL
# ==========================================
class EdgeBasedBAGraphGNN(nn.Module):

    def __init__(self, input_dim, hidden_dim, K_iterations):
        super(EdgeBasedBAGraphGNN, self).__init__()
        self.K = K_iterations
        self.hidden_dim = hidden_dim

        self.edge_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.1),
        )

        self.W_msg  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_self = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.activation = nn.LeakyReLU(0.1)
        self.dropout    = nn.Dropout(p=0.3)

        self.node_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, B, norm_A_L):
        M = B.size(1)
        if M == 0:

          h_node = torch.zeros(x.size(0), self.hidden_dim, device=x.device)
          return self.sigmoid(self.node_classifier(h_node))

        # Average incident node features to initialize edge features
        e_init = (B.t() @ x) / 2.0
        e = self.edge_proj(e_init)

        # Message passing over the line graph
        for _ in range(self.K):
            msg = torch.matmul(norm_A_L, self.W_msg(e))
            e_self = self.W_self(e)

            e_new  = self.activation(e_self + msg)
            e_new  = self.layer_norm(e_new)
            e      = self.dropout(e_new)

        # Aggregate edge representations back to nodes
        h_node = B @ e

        return self.sigmoid(self.node_classifier(h_node))

# ==========================================
# 6. VALIDITY CHECK
# ==========================================
def is_valid_vertex_cover(graph, selected_nodes):
    selected = set(selected_nodes)
    return all(u in selected or v in selected for u, v in graph.edges())

# ==========================================
# 7. POST-PROCESSING — GREEDY COVER REPAIR
# ==========================================
def greedy_cover_from_probs(probs, edges, num_nodes):
    """
    Add nodes greedily (highest GNN probability first) until
    all edges are covered. Guarantees a valid vertex cover
    regardless of the raw GNN output quality.
    """
    sorted_nodes   = sorted(range(len(probs)), key=lambda k: probs[k], reverse=True)
    selected       = []
    remaining      = list(edges)

    for node in sorted_nodes:
        selected.append(node)
        remaining = [e for e in remaining
                     if e[0] not in selected and e[1] not in selected]
        if not remaining:
            break

    return selected

# ==========================================
# 8. MAIN EXECUTION
# ==========================================
INPUT_DIM  = 5
HIDDEN_DIM = 16
K_HOPS     = 4

print("Loading datasets...")
train_data = load_dataset('train')
test_data  = load_dataset('test')

# ---- Build Training Pairs ----
print("Extracting features and labels for watts-strogatz graphs...")
training_pairs = []
for data in train_data:
    if data['type'] != 'watts-strogatz':
        continue

    x           = extract_ba_features(data['graph'])
    B, norm_A_L = get_edge_structures(data['graph'])
    y           = get_ground_truth_vertex_cover(data['graph']).unsqueeze(1)
    training_pairs.append((x, B, norm_A_L, y))

print(f"Found {len(training_pairs)} watts-strogatz training graphs.")

# ---- Train ----
model     = EdgeBasedBAGraphGNN(INPUT_DIM, HIDDEN_DIM, K_HOPS)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
criterion = nn.BCELoss()

epochs = 1000
print(f"\nStarting training for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, B, norm_A_L, y in training_pairs:
            
            # Skip empty graph topologies to prevent detached NaN losses
            #if x.size(0) == 0 or B.size(1) == 0:
            #    continue
                
            optimizer.zero_grad()
            
            pred      = model(x, B, norm_A_L)
            loss_bce  = criterion(pred, y)
    
            loss = loss_bce + 0.05 * pred.mean()
    
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

    scheduler.step()

    if (epoch + 1) % 20 == 0:
        avg = total_loss / max(1, len(training_pairs))
        lr  = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg:.4f} | LR: {lr:.6f}")

# ---- Test ----
print("\n" + "=" * 45)
print("TESTING ON UNSEEN watts-strogatz GRAPHS")
print("=" * 45)

model.eval()

valid_count   = 0
better_count  = 0
equal_count   = 0
total         = 0
approx_ratios = []

for data in test_data:
    if data['type'] != 'watts-strogatz':
        continue

    x           = extract_ba_features(data['graph'])
    B, norm_A_L = get_edge_structures(data['graph'])

    with torch.no_grad():
        probs = model(x, B, norm_A_L)

    node_probs     = probs.view(-1).tolist()
    selected_nodes = greedy_cover_from_probs(
        node_probs, list(data['graph'].edges()), data['num_nodes']
    )

    predicted_size = len(selected_nodes)
    is_valid       = is_valid_vertex_cover(data['graph'], selected_nodes)

    # Baseline: NetworkX 2-approximation
    approx_vc   = nx.algorithms.approximation.min_weighted_vertex_cover(data['graph'])
    approx_size = len(approx_vc)

    total += 1
    if is_valid:
        valid_count += 1
        ratio = predicted_size / approx_size
        approx_ratios.append(ratio)

    beats_baseline_strict = is_valid and (predicted_size < approx_size)
    matches_baseline      = is_valid and (predicted_size == approx_size)

    if beats_baseline_strict:
        better_count += 1
    if matches_baseline:
        equal_count += 1

    verdict = "beats baseline" if beats_baseline_strict else (
              "matches baseline" if matches_baseline else (
              "valid but larger" if is_valid else "INVALID COVER"))

    print(f"Graph ID : {data['id']} | Nodes: {data['num_nodes']}")
    print(f" > Valid Cover   : {is_valid}")
    print(f" > Predicted Size: {predicted_size}")
    print(f" > Baseline Size : {approx_size}  ({verdict})")
    print(f" > Selected Nodes: {selected_nodes}")
    print("-" * 40)

# ---- Final Report ----
print("\n" + "=" * 45)
print("FINAL RESULTS")
print("=" * 45)
validity     = (valid_count  / total * 100) if total > 0 else 0
beat_pct     = (better_count / total * 100) if total > 0 else 0
match_pct    = (equal_count  / total * 100) if total > 0 else 0
avg_ratio    = np.mean(approx_ratios) if approx_ratios else float('inf')

print(f"Total Graphs Tested   : {total}")
print(f"Valid Covers          : {valid_count}/{total}  ({validity:.1f}%)")
print(f"Beats Baseline        : {better_count}/{total}  ({beat_pct:.1f}%)")
print(f"Matches Baseline      : {equal_count}/{total}  ({match_pct:.1f}%)")
print(f"Avg Approx Ratio      : {avg_ratio:.3f}  (vs baseline; <1.0 means smaller cover)")
print(f"Note: Baseline is NetworkX 2-approximation, NOT true optimum.")
print("=" * 45)

Loading datasets...
Extracting features and labels for watts-strogatz graphs...
Found 1000 watts-strogatz training graphs.

Starting training for 1000 epochs...
Epoch 20/1000 | Loss: 0.5219 | LR: 0.005000
Epoch 40/1000 | Loss: 0.5003 | LR: 0.005000
Epoch 60/1000 | Loss: 0.4951 | LR: 0.005000
Epoch 80/1000 | Loss: 0.5049 | LR: 0.005000
Epoch 100/1000 | Loss: 0.4921 | LR: 0.002500
Epoch 120/1000 | Loss: 0.4885 | LR: 0.002500
Epoch 140/1000 | Loss: 0.4856 | LR: 0.002500
Epoch 160/1000 | Loss: 0.4881 | LR: 0.002500
Epoch 180/1000 | Loss: 0.4920 | LR: 0.002500
Epoch 200/1000 | Loss: 0.4824 | LR: 0.001250
Epoch 220/1000 | Loss: 0.4806 | LR: 0.001250
Epoch 240/1000 | Loss: 0.4774 | LR: 0.001250
Epoch 260/1000 | Loss: 0.4825 | LR: 0.001250
Epoch 280/1000 | Loss: 0.4821 | LR: 0.001250
Epoch 300/1000 | Loss: 0.4792 | LR: 0.000625
Epoch 320/1000 | Loss: 0.4741 | LR: 0.000625
Epoch 340/1000 | Loss: 0.4736 | LR: 0.000625
Epoch 360/1000 | Loss: 0.4734 | LR: 0.000625
Epoch 380/1000 | Loss: 0.4753 | L

# **GATv2 (Attention-based GNN) + Positional**

In [4]:
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
import pickle

# ==========================================
# 1. DATASET LOADING
# ==========================================
def load_dataset(folder_name):
    dataset = []
    if not os.path.exists(folder_name):
        print(f"Warning: folder '{folder_name}' not found.")
        return dataset
    for filename in sorted(os.listdir(folder_name)):
        if filename.endswith('.pkl'):
            with open(os.path.join(folder_name, filename), 'rb') as f:
                data = pickle.load(f)
                G = nx.Graph()
                G.add_nodes_from(range(data['num_vertices']))
                G.add_edges_from(data['edge_list'])
                dataset.append({
                    'id'        : data['id'],
                    'type'      : data['type'],
                    'graph'     : G,
                    'adj_matrix': torch.tensor(data['adj_matrix'], dtype=torch.float32),
                    'num_nodes' : data['num_vertices']
                })
    return dataset

# ==========================================
# 2. EDGE STRUCTURE EXTRACTION (LINE GRAPH)
# ==========================================
def get_edge_structures(graph):
    edges = list(graph.edges())
    N = graph.number_of_nodes()
    M = len(edges)

    B = torch.zeros((N, M), dtype=torch.float32)
    for j, (u, v) in enumerate(edges):
        B[u, j] = 1.0
        B[v, j] = 1.0

    if M == 0:
        return B, torch.zeros((0, 0), dtype=torch.float32)

    A_L = B.t() @ B - 2.0 * torch.eye(M)
    A_L = torch.clamp(A_L, min=0.0)

    A_L_self = A_L + torch.eye(M)
    D_L = A_L_self.sum(dim=1)
    D_L_inv_sqrt = torch.diag(1.0 / torch.sqrt(D_L.clamp(min=1e-8)))
    norm_A_L = D_L_inv_sqrt @ A_L_self @ D_L_inv_sqrt

    return B, norm_A_L

# ==========================================
# 3. GROUND TRUTH — GREEDY + REDUCTION
# ==========================================
def get_ground_truth_vertex_cover(graph):
    G_copy = graph.copy()
    cover_set = set()
    while G_copy.number_of_edges() > 0:
        node = max(G_copy.degree(), key=lambda x: x[1])[0]
        cover_set.add(node)
        G_copy.remove_node(node)

    improved = True
    while improved:
        improved = False
        for node in list(cover_set):
            cover_set.remove(node)
            still_valid = all(
                u in cover_set or v in cover_set
                for u, v in graph.edges()
            )
            if still_valid:
                improved = True
            else:
                cover_set.add(node)

    label = torch.zeros(graph.number_of_nodes(), dtype=torch.float32)
    for node in cover_set:
        label[node] = 1.0
    return label

# ==========================================
# 4. LAPLACIAN POSITIONAL ENCODING
# ==========================================
def compute_laplacian_pe(adj_matrix: torch.Tensor, pe_dim=4) -> torch.Tensor:
    A = adj_matrix
    N = A.size(0)
    D_vec = A.sum(dim=1)
    L = torch.diag(D_vec) - A

    D_inv_sqrt = torch.diag(1.0 / torch.sqrt(D_vec.clamp(min=1e-8)))
    L_sym = D_inv_sqrt @ L @ D_inv_sqrt

    try:
        eigenvalues, eigenvectors = torch.linalg.eigh(L_sym)
        pe = eigenvectors[:, 1:pe_dim+1]
        if pe.size(1) < pe_dim:
            padding = torch.zeros((N, pe_dim - pe.size(1)), device=pe.device)
            pe = torch.cat([pe, padding], dim=1)
    except Exception:
        pe = torch.zeros((N, pe_dim), dtype=torch.float32)

    return pe

# ==========================================
# 5. BA-AWARE FEATURE EXTRACTION
# ==========================================
def extract_ba_features(graph):
    n = graph.number_of_nodes()
    degrees = dict(graph.degree())
    max_deg = max(degrees.values()) if max(degrees.values()) > 0 else 1

    clustering     = nx.clustering(graph)
    triangles      = nx.triangles(graph)
    max_tri        = max(triangles.values()) if max(triangles.values()) > 0 else 1
    deg_centrality = nx.degree_centrality(graph)

    avg_neighbor_deg = {}
    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        avg_neighbor_deg[node] = (
            np.mean([degrees[nb] for nb in neighbors]) / max_deg
            if neighbors else 0.0
        )

    features = []
    for node in range(n):
        features.append([
            degrees[node] / max_deg,
            clustering[node],
            triangles[node] / max_tri,
            avg_neighbor_deg[node],
            deg_centrality[node],
        ])

    return torch.tensor(features, dtype=torch.float32)

# ==========================================
# 6. DENSE GATv2 LAYER
# ==========================================
class DenseGATv2Layer(nn.Module):
    def __init__(self, in_features, out_features, alpha=0.2):
        super(DenseGATv2Layer, self).__init__()
        self.W_l = nn.Linear(in_features, out_features, bias=False)
        self.W_r = nn.Linear(in_features, out_features, bias=False)
        self.a = nn.Parameter(torch.Tensor(out_features, 1))
        self.leakyrelu = nn.LeakyReLU(alpha)

        nn.init.xavier_uniform_(self.W_l.weight.data)
        nn.init.xavier_uniform_(self.W_r.weight.data)
        nn.init.xavier_uniform_(self.a.data)

    def forward(self, h, adj):
        Wh_l = self.W_l(h)
        Wh_r = self.W_r(h)

        Wh_i = Wh_l.unsqueeze(1)
        Wh_j = Wh_r.unsqueeze(0)

        e = torch.matmul(self.leakyrelu(Wh_i + Wh_j), self.a).squeeze(2)

        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 1e-6, e, zero_vec)
        attention = torch.softmax(attention, dim=1)

        h_prime = torch.matmul(attention, Wh_r)
        return h_prime

# ==========================================
# 7. EDGE-BASED GNN MODEL (GATv2)
# ==========================================
class EdgeBasedBAGraphGNN(nn.Module):

    def __init__(self, input_dim, hidden_dim, K_iterations):
        super(EdgeBasedBAGraphGNN, self).__init__()
        self.K = K_iterations
        self.hidden_dim = hidden_dim

        self.edge_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.1),
        )

        self.gat_layers = nn.ModuleList([
            DenseGATv2Layer(hidden_dim, hidden_dim) for _ in range(self.K)
        ])

        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(self.K)
        ])

        self.dropout = nn.Dropout(p=0.3)

        self.node_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, B, norm_A_L):
        M = B.size(1)
        if M == 0:
          h_node = torch.zeros(x.size(0), self.hidden_dim, device=x.device)
          return self.sigmoid(self.node_classifier(h_node))

        e_init = (B.t() @ x) / 2.0
        e = self.edge_proj(e_init)

        for i in range(self.K):
            msg_e = self.gat_layers[i](e, norm_A_L)
            e_new = self.layer_norms[i](e + msg_e)
            e     = self.dropout(e_new)

        h_node = B @ e

        return self.sigmoid(self.node_classifier(h_node))

# ==========================================
# 8. VALIDITY CHECK & POST-PROCESSING
# ==========================================
def is_valid_vertex_cover(graph, selected_nodes):
    selected = set(selected_nodes)
    return all(u in selected or v in selected for u, v in graph.edges())

def greedy_cover_from_probs(probs, edges, num_nodes):
    sorted_nodes   = sorted(range(len(probs)), key=lambda k: probs[k], reverse=True)
    selected       = []
    remaining      = list(edges)

    for node in sorted_nodes:
        selected.append(node)
        remaining = [e for e in remaining
                     if e[0] not in selected and e[1] not in selected]
        if not remaining:
            break

    return selected

# ==========================================
# 9. MAIN EXECUTION
# ==========================================
PE_DIM     = 4
INPUT_DIM  = 5 + PE_DIM
HIDDEN_DIM = 16
K_HOPS     = 4

print("Loading datasets...")
train_data = load_dataset('train')
test_data  = load_dataset('test')

# ---- Build Training Pairs ----
print("Extracting features and labels for watts-strogatz graphs...")
training_pairs = []
for data in train_data:
    if data['type'] != 'watts-strogatz':
        continue

    x_features  = extract_ba_features(data['graph'])
    pe_features = compute_laplacian_pe(data['adj_matrix'], pe_dim=PE_DIM)
    x           = torch.cat([x_features, pe_features], dim=1)

    B, norm_A_L = get_edge_structures(data['graph'])
    y           = get_ground_truth_vertex_cover(data['graph']).unsqueeze(1)
    training_pairs.append((x, B, norm_A_L, y))

print(f"Found {len(training_pairs)} watts-strogatz training graphs.")

# ---- Train ----
model     = EdgeBasedBAGraphGNN(INPUT_DIM, HIDDEN_DIM, K_HOPS)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
criterion = nn.BCELoss()

epochs = 1000
print(f"\nStarting training for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, B, norm_A_L, y in training_pairs:
        optimizer.zero_grad()

        pred      = model(x, B, norm_A_L)
        loss_bce  = criterion(pred, y)

        loss = loss_bce + 0.05 * pred.mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    if (epoch + 1) % 20 == 0:
        avg = total_loss / max(1, len(training_pairs))
        lr  = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg:.4f} | LR: {lr:.6f}")

# ---- Test ----
print("\n" + "=" * 45)
print("TESTING ON UNSEEN watts-strogatz GRAPHS")
print("=" * 45)

model.eval()

valid_count   = 0
better_count  = 0
equal_count   = 0
total         = 0
approx_ratios = []

for data in test_data:
    if data['type'] != 'watts-strogatz':
        continue

    x_features  = extract_ba_features(data['graph'])
    pe_features = compute_laplacian_pe(data['adj_matrix'], pe_dim=PE_DIM)
    x           = torch.cat([x_features, pe_features], dim=1)

    B, norm_A_L = get_edge_structures(data['graph'])

    with torch.no_grad():
        probs = model(x, B, norm_A_L)

    node_probs     = probs.view(-1).tolist()
    selected_nodes = greedy_cover_from_probs(
        node_probs, list(data['graph'].edges()), data['num_nodes']
    )

    predicted_size = len(selected_nodes)
    is_valid       = is_valid_vertex_cover(data['graph'], selected_nodes)

    approx_vc   = nx.algorithms.approximation.min_weighted_vertex_cover(data['graph'])
    approx_size = len(approx_vc)

    total += 1
    if is_valid:
        valid_count += 1
        ratio = predicted_size / approx_size
        approx_ratios.append(ratio)

    beats_baseline_strict = is_valid and (predicted_size < approx_size)
    matches_baseline      = is_valid and (predicted_size == approx_size)

    if beats_baseline_strict:
        better_count += 1
    if matches_baseline:
        equal_count += 1

    verdict = "beats baseline" if beats_baseline_strict else (
              "matches baseline" if matches_baseline else (
              "valid but larger" if is_valid else "INVALID COVER"))

    print(f"Graph ID : {data['id']} | Nodes: {data['num_nodes']}")
    print(f" > Valid Cover   : {is_valid}")
    print(f" > Predicted Size: {predicted_size}")
    print(f" > Baseline Size : {approx_size}  ({verdict})")
    print(f" > Selected Nodes: {selected_nodes}")
    print("-" * 40)

# ---- Final Report ----
print("\n" + "=" * 45)
print("FINAL RESULTS")
print("=" * 45)
validity     = (valid_count  / total * 100) if total > 0 else 0
beat_pct     = (better_count / total * 100) if total > 0 else 0
match_pct    = (equal_count  / total * 100) if total > 0 else 0
avg_ratio    = np.mean(approx_ratios) if approx_ratios else float('inf')

print(f"Total Graphs Tested   : {total}")
print(f"Valid Covers          : {valid_count}/{total}  ({validity:.1f}%)")
print(f"Beats Baseline        : {better_count}/{total}  ({beat_pct:.1f}%)")
print(f"Matches Baseline      : {equal_count}/{total}  ({match_pct:.1f}%)")
print(f"Avg Approx Ratio      : {avg_ratio:.3f}  (vs baseline; <1.0 means smaller cover)")
print(f"Note: Baseline is NetworkX 2-approximation, NOT true optimum.")
print("=" * 45)

Loading datasets...
Extracting features and labels for watts-strogatz graphs...
Found 1000 watts-strogatz training graphs.

Starting training for 1000 epochs...
Epoch 20/1000 | Loss: 0.5697 | LR: 0.005000
Epoch 40/1000 | Loss: 0.5697 | LR: 0.005000
Epoch 60/1000 | Loss: 0.5688 | LR: 0.005000
Epoch 80/1000 | Loss: 0.5684 | LR: 0.005000
Epoch 100/1000 | Loss: 0.5682 | LR: 0.002500
Epoch 120/1000 | Loss: 0.5663 | LR: 0.002500
Epoch 140/1000 | Loss: 0.5409 | LR: 0.002500
Epoch 160/1000 | Loss: 0.5148 | LR: 0.002500
Epoch 180/1000 | Loss: 0.5059 | LR: 0.002500
Epoch 200/1000 | Loss: 0.5107 | LR: 0.001250
Epoch 220/1000 | Loss: 0.4934 | LR: 0.001250
Epoch 240/1000 | Loss: 0.4872 | LR: 0.001250
Epoch 260/1000 | Loss: 0.4894 | LR: 0.001250
Epoch 280/1000 | Loss: 0.4886 | LR: 0.001250
Epoch 300/1000 | Loss: 0.4880 | LR: 0.000625
Epoch 320/1000 | Loss: 0.4790 | LR: 0.000625
Epoch 340/1000 | Loss: 0.4847 | LR: 0.000625
Epoch 360/1000 | Loss: 0.4727 | LR: 0.000625
Epoch 380/1000 | Loss: 0.4778 | L

# **GIN (Graph Isomorphism Network) + Structural Features**

In [5]:
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
import pickle

# ==========================================
# 1. DATASET LOADING
# ==========================================
def load_dataset(folder_name):
    dataset = []
    if not os.path.exists(folder_name):
        print(f"Warning: folder '{folder_name}' not found.")
        return dataset
    for filename in sorted(os.listdir(folder_name)):
        if filename.endswith('.pkl'):
            with open(os.path.join(folder_name, filename), 'rb') as f:
                data = pickle.load(f)
                G = nx.Graph()
                G.add_nodes_from(range(data['num_vertices']))
                G.add_edges_from(data['edge_list'])
                dataset.append({
                    'id'        : data['id'],
                    'type'      : data['type'],
                    'graph'     : G,
                    'adj_matrix': torch.tensor(data['adj_matrix'], dtype=torch.float32),
                    'num_nodes' : data['num_vertices']
                })
    return dataset

# ==========================================
# 2. EDGE STRUCTURE EXTRACTION (LINE GRAPH)
# ==========================================
def get_edge_structures(graph):
    edges = list(graph.edges())
    N = graph.number_of_nodes()
    M = len(edges)

    B = torch.zeros((N, M), dtype=torch.float32)
    for j, (u, v) in enumerate(edges):
        B[u, j] = 1.0
        B[v, j] = 1.0

    if M == 0:
        return B, torch.zeros((0, 0), dtype=torch.float32)

    # Line graph adjacency A_L = B^T * B - 2I
    # For GIN, we strictly use the unnormalized adjacency matrix
    # to maintain the injective sum-aggregation properties.
    A_L = B.t() @ B - 2.0 * torch.eye(M)
    A_L = torch.clamp(A_L, min=0.0)

    return B, A_L

# ==========================================
# 3. GROUND TRUTH — GREEDY + REDUCTION
# ==========================================
def get_ground_truth_vertex_cover(graph):
    G_copy = graph.copy()
    cover_set = set()
    while G_copy.number_of_edges() > 0:
        node = max(G_copy.degree(), key=lambda x: x[1])[0]
        cover_set.add(node)
        G_copy.remove_node(node)

    improved = True
    while improved:
        improved = False
        for node in list(cover_set):
            cover_set.remove(node)
            still_valid = all(
                u in cover_set or v in cover_set
                for u, v in graph.edges()
            )
            if still_valid:
                improved = True
            else:
                cover_set.add(node)

    label = torch.zeros(graph.number_of_nodes(), dtype=torch.float32)
    for node in cover_set:
        label[node] = 1.0
    return label

# ==========================================
# 4. LAPLACIAN POSITIONAL ENCODING
# ==========================================
def compute_laplacian_pe(adj_matrix: torch.Tensor, pe_dim=4) -> torch.Tensor:
    A = adj_matrix
    N = A.size(0)
    D_vec = A.sum(dim=1)
    L = torch.diag(D_vec) - A

    D_inv_sqrt = torch.diag(1.0 / torch.sqrt(D_vec.clamp(min=1e-8)))
    L_sym = D_inv_sqrt @ L @ D_inv_sqrt

    try:
        eigenvalues, eigenvectors = torch.linalg.eigh(L_sym)
        pe = eigenvectors[:, 1:pe_dim+1]
        if pe.size(1) < pe_dim:
            padding = torch.zeros((N, pe_dim - pe.size(1)), device=pe.device)
            pe = torch.cat([pe, padding], dim=1)
    except Exception:
        pe = torch.zeros((N, pe_dim), dtype=torch.float32)

    return pe

# ==========================================
# 5. BA-AWARE FEATURE EXTRACTION
# ==========================================
def extract_ba_features(graph):
    n = graph.number_of_nodes()
    degrees = dict(graph.degree())
    max_deg = max(degrees.values()) if max(degrees.values()) > 0 else 1

    clustering     = nx.clustering(graph)
    triangles      = nx.triangles(graph)
    max_tri        = max(triangles.values()) if max(triangles.values()) > 0 else 1
    deg_centrality = nx.degree_centrality(graph)

    avg_neighbor_deg = {}
    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        avg_neighbor_deg[node] = (
            np.mean([degrees[nb] for nb in neighbors]) / max_deg
            if neighbors else 0.0
        )

    features = []
    for node in range(n):
        features.append([
            degrees[node] / max_deg,
            clustering[node],
            triangles[node] / max_tri,
            avg_neighbor_deg[node],
            deg_centrality[node],
        ])

    return torch.tensor(features, dtype=torch.float32)

# ==========================================
# 6. GIN LAYER
# ==========================================
class GINLayer(nn.Module):
    def __init__(self, hidden_dim):
        super(GINLayer, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.1)
        )
        self.eps = nn.Parameter(torch.zeros(1))

    def forward(self, h, adj):
        msg = torch.matmul(adj, h)
        out = (1.0 + self.eps) * h + msg
        return self.mlp(out)

# ==========================================
# 7. EDGE-BASED GNN MODEL (GIN)
# ==========================================
class EdgeBasedBAGraphGNN(nn.Module):

    def __init__(self, input_dim, hidden_dim, K_iterations):
        super(EdgeBasedBAGraphGNN, self).__init__()
        self.K = K_iterations
        self.hidden_dim = hidden_dim
        
        self.edge_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.LeakyReLU(0.1),
        )

        self.gin_layers = nn.ModuleList([
            GINLayer(hidden_dim) for _ in range(self.K)
        ])

        self.dropout = nn.Dropout(p=0.3)

        self.node_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, B, A_L):
        M = B.size(1)
        if M == 0:
            h_node = torch.zeros(x.size(0), self.hidden_dim, device=x.device)
            return self.sigmoid(self.node_classifier(h_node))

        e_init = (B.t() @ x) / 2.0
        e = self.edge_proj(e_init)

        for i in range(self.K):
            e = self.gin_layers[i](e, A_L)
            e = self.dropout(e)

        h_node = B @ e

        return self.sigmoid(self.node_classifier(h_node))

# ==========================================
# 8. VALIDITY CHECK & POST-PROCESSING
# ==========================================
def is_valid_vertex_cover(graph, selected_nodes):
    selected = set(selected_nodes)
    return all(u in selected or v in selected for u, v in graph.edges())

def greedy_cover_from_probs(probs, edges, num_nodes):
    sorted_nodes   = sorted(range(len(probs)), key=lambda k: probs[k], reverse=True)
    selected       = []
    remaining      = list(edges)

    for node in sorted_nodes:
        selected.append(node)
        remaining = [e for e in remaining
                     if e[0] not in selected and e[1] not in selected]
        if not remaining:
            break

    return selected

# ==========================================
# 9. MAIN EXECUTION
# ==========================================
PE_DIM     = 4
INPUT_DIM  = 5 + PE_DIM
HIDDEN_DIM = 16
K_HOPS     = 4

print("Loading datasets...")
train_data = load_dataset('train')
test_data  = load_dataset('test')

# ---- Build Training Pairs ----
print("Extracting features and labels for watts-strogatz graphs...")
training_pairs = []
for data in train_data:
    if data['type'] != 'watts-strogatz':
        continue

    x_features  = extract_ba_features(data['graph'])
    pe_features = compute_laplacian_pe(data['adj_matrix'], pe_dim=PE_DIM)
    x           = torch.cat([x_features, pe_features], dim=1)

    B, A_L      = get_edge_structures(data['graph'])
    y           = get_ground_truth_vertex_cover(data['graph']).unsqueeze(1)
    training_pairs.append((x, B, A_L, y))

print(f"Found {len(training_pairs)} ER training graphs.")

# ---- Train ----
model     = EdgeBasedBAGraphGNN(INPUT_DIM, HIDDEN_DIM, K_HOPS)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
criterion = nn.BCELoss()

epochs = 1000
print(f"\nStarting training for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, B, A_L, y in training_pairs:
        optimizer.zero_grad()

        pred      = model(x, B, A_L)
        loss_bce  = criterion(pred, y)

        loss = loss_bce + 0.05 * pred.mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    print(f"Epoch: {epoch}")
    if (epoch + 1) % 20 == 0:
        avg = total_loss / max(1, len(training_pairs))
        lr  = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg:.4f} | LR: {lr:.6f}")

# ---- Test ----
print("\n" + "=" * 45)
print("TESTING ON UNSEEN watts-strogatz GRAPHS")
print("=" * 45)

model.eval()

valid_count   = 0
better_count  = 0
equal_count   = 0
total         = 0
approx_ratios = []

for data in test_data:
    if data['type'] != 'watts-strogatz':
        continue

    x_features  = extract_ba_features(data['graph'])
    pe_features = compute_laplacian_pe(data['adj_matrix'], pe_dim=PE_DIM)
    x           = torch.cat([x_features, pe_features], dim=1)

    B, A_L      = get_edge_structures(data['graph'])

    with torch.no_grad():
        probs = model(x, B, A_L)

    node_probs     = probs.view(-1).tolist()
    selected_nodes = greedy_cover_from_probs(
        node_probs, list(data['graph'].edges()), data['num_nodes']
    )

    predicted_size = len(selected_nodes)
    is_valid       = is_valid_vertex_cover(data['graph'], selected_nodes)

    approx_vc   = nx.algorithms.approximation.min_weighted_vertex_cover(data['graph'])
    approx_size = len(approx_vc)

    total += 1
    if is_valid:
        valid_count += 1
        ratio = predicted_size / approx_size
        approx_ratios.append(ratio)

    beats_baseline_strict = is_valid and (predicted_size < approx_size)
    matches_baseline      = is_valid and (predicted_size == approx_size)

    if beats_baseline_strict:
        better_count += 1
    if matches_baseline:
        equal_count += 1

    verdict = "beats baseline" if beats_baseline_strict else (
              "matches baseline" if matches_baseline else (
              "valid but larger" if is_valid else "INVALID COVER"))

    print(f"Graph ID : {data['id']} | Nodes: {data['num_nodes']}")
    print(f" > Valid Cover   : {is_valid}")
    print(f" > Predicted Size: {predicted_size}")
    print(f" > Baseline Size : {approx_size}  ({verdict})")
    print(f" > Selected Nodes: {selected_nodes}")
    print("-" * 40)

# ---- Final Report ----
print("\n" + "=" * 45)
print("FINAL RESULTS")
print("=" * 45)
validity     = (valid_count  / total * 100) if total > 0 else 0
beat_pct     = (better_count / total * 100) if total > 0 else 0
match_pct    = (equal_count  / total * 100) if total > 0 else 0
avg_ratio    = np.mean(approx_ratios) if approx_ratios else float('inf')

print(f"Total Graphs Tested   : {total}")
print(f"Valid Covers          : {valid_count}/{total}  ({validity:.1f}%)")
print(f"Beats Baseline        : {better_count}/{total}  ({beat_pct:.1f}%)")
print(f"Matches Baseline      : {equal_count}/{total}  ({match_pct:.1f}%)")
print(f"Avg Approx Ratio      : {avg_ratio:.3f}  (vs baseline; <1.0 means smaller cover)")
print(f"Note: Baseline is NetworkX 2-approximation, NOT true optimum.")
print("=" * 45)

Loading datasets...
Extracting features and labels for watts-strogatz graphs...
Found 1000 ER training graphs.

Starting training for 1000 epochs...
Epoch: 0
Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
Epoch: 6
Epoch: 7
Epoch: 8
Epoch: 9
Epoch: 10
Epoch: 11
Epoch: 12
Epoch: 13
Epoch: 14
Epoch: 15
Epoch: 16
Epoch: 17
Epoch: 18
Epoch: 19
Epoch 20/1000 | Loss: 0.5427 | LR: 0.005000
Epoch: 20
Epoch: 21
Epoch: 22
Epoch: 23
Epoch: 24
Epoch: 25
Epoch: 26
Epoch: 27
Epoch: 28
Epoch: 29
Epoch: 30
Epoch: 31
Epoch: 32
Epoch: 33
Epoch: 34
Epoch: 35
Epoch: 36
Epoch: 37
Epoch: 38
Epoch: 39
Epoch 40/1000 | Loss: 0.5426 | LR: 0.005000
Epoch: 40
Epoch: 41
Epoch: 42
Epoch: 43
Epoch: 44
Epoch: 45
Epoch: 46
Epoch: 47
Epoch: 48
Epoch: 49
Epoch: 50
Epoch: 51
Epoch: 52
Epoch: 53
Epoch: 54
Epoch: 55
Epoch: 56
Epoch: 57
Epoch: 58
Epoch: 59
Epoch 60/1000 | Loss: 0.5437 | LR: 0.005000
Epoch: 60
Epoch: 61
Epoch: 62
Epoch: 63
Epoch: 64
Epoch: 65
Epoch: 66
Epoch: 67
Epoch: 68
Epoch: 69
Epoch: 70
Epoch: 71
Epoch: 72

# **S2V-DQN (Structure2Vec GNN with Deep Q-Learning)**

In [6]:
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
import pickle
import random
from collections import deque

# ==========================================
# 0. DEVICE CONFIGURATION
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing computations on device: {device}")

# ==========================================
# 1. DATASET LOADING
# ==========================================
def load_dataset(folder_name):
    dataset = []
    if not os.path.exists(folder_name):
        print(f"Warning: folder '{folder_name}' not found.")
        return dataset
    for filename in sorted(os.listdir(folder_name)):
        if filename.endswith('.pkl'):
            with open(os.path.join(folder_name, filename), 'rb') as f:
                data = pickle.load(f)
                G = nx.Graph()
                G.add_nodes_from(range(data['num_vertices']))
                G.add_edges_from(data['edge_list'])
                dataset.append({
                    'id'        : data['id'],
                    'type'      : data['type'],
                    'graph'     : G,
                    # Allocate adjacency matrix directly to the selected device
                    'adj_matrix': torch.tensor(data['adj_matrix'], dtype=torch.float32, device=device),
                    'num_nodes' : data['num_vertices']
                })
    return dataset

# ==========================================
# 2. NORMALISED ADJACENCY
# ==========================================
def normalized_adj(adj_matrix: torch.Tensor) -> torch.Tensor:
    # Ensure the identity matrix is allocated on the same device as the adjacency matrix
    A = adj_matrix + torch.eye(adj_matrix.size(0), device=adj_matrix.device)
    D = A.sum(dim=1)
    D_inv_sqrt = torch.diag(1.0 / torch.sqrt(D.clamp(min=1e-8)))
    return D_inv_sqrt @ A @ D_inv_sqrt

# ==========================================
# 3. BA-AWARE FEATURE EXTRACTION (STATIC)
# ==========================================
def extract_ba_features(graph):
    n = graph.number_of_nodes()
    degrees = dict(graph.degree())
    max_deg = max(degrees.values()) if max(degrees.values()) > 0 else 1

    clustering     = nx.clustering(graph)
    triangles      = nx.triangles(graph)
    max_tri        = max(triangles.values()) if max(triangles.values()) > 0 else 1
    deg_centrality = nx.degree_centrality(graph)

    avg_neighbor_deg = {}
    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        avg_neighbor_deg[node] = (
            np.mean([degrees[nb] for nb in neighbors]) / max_deg
            if neighbors else 0.0
        )

    features = []
    for node in range(n):
        features.append([
            degrees[node] / max_deg,
            clustering[node],
            triangles[node] / max_tri,
            avg_neighbor_deg[node],
            deg_centrality[node],
        ])

    # Allocate extracted feature tensor directly to the selected device
    return torch.tensor(features, dtype=torch.float32, device=device)

# ==========================================
# 4. REINFORCEMENT LEARNING ENVIRONMENT
# ==========================================
class MVCEnvironment:
    def __init__(self, graph):
        self.graph = graph
        self.num_nodes = graph.number_of_nodes()
        self.edges = set(
            (u, v) if u < v else (v, u) for u, v in graph.edges()
        )
        self.reset()

    def reset(self):
        self.covered_edges = set()
        self.cover = set()
        # Initialize the dynamic state mask tensor on the selected device
        self.state_mask = torch.zeros((self.num_nodes, 1), dtype=torch.float32, device=device)
        return self.state_mask.clone()

    def step(self, action):
        self.cover.add(action)
        self.state_mask[action, 0] = 1.0

        for neighbor in self.graph.neighbors(action):
            edge = (action, neighbor) if action < neighbor else (neighbor, action)
            if edge in self.edges:
                self.covered_edges.add(edge)

        done = len(self.covered_edges) == len(self.edges)
        reward = -1.0
        return self.state_mask.clone(), reward, done

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, transition):
        self.buffer.append(transition)

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

# ==========================================
# 5. STRUCTURE2VEC Q-NETWORK
# ==========================================
class Structure2VecDQN(nn.Module):
    def __init__(self, input_dim, hidden_dim, K_iterations):
        super(Structure2VecDQN, self).__init__()
        self.K = K_iterations

        self.proj = nn.Linear(input_dim + 1, hidden_dim)

        self.W_msg = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_upd = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self.q_node  = nn.Linear(hidden_dim, hidden_dim)
        self.q_graph = nn.Linear(hidden_dim, hidden_dim)
        self.q_out   = nn.Linear(hidden_dim, 1)

    def forward(self, static_features, state_mask, norm_adj):
        x = torch.cat([static_features, state_mask], dim=-1)
        h = torch.relu(self.proj(x))

        for _ in range(self.K):
            msg = torch.matmul(norm_adj, self.W_msg(h))
            h = torch.relu(self.W_upd(h) + msg)

        graph_embed = h.sum(dim=0, keepdim=True)

        q_n = self.q_node(h)
        q_g = self.q_graph(graph_embed)
        q_vals = self.q_out(torch.relu(q_n + q_g))

        return q_vals.squeeze(-1)

# ==========================================
# 6. RL UTILITIES
# ==========================================
def select_action(q_vals, state_mask, epsilon):
    valid_actions = (state_mask.squeeze(-1) == 0).nonzero(as_tuple=True)[0].tolist()
    if random.random() < epsilon:
        return random.choice(valid_actions)
    else:
        q_valid = q_vals.clone()
        q_valid[state_mask.squeeze(-1) == 1] = -float('inf')
        return torch.argmax(q_valid).item()

def is_valid_vertex_cover(graph, selected_nodes):
    selected = set(selected_nodes)
    return all(u in selected or v in selected for u, v in graph.edges())

# ==========================================
# 7. MAIN EXECUTION
# ==========================================
INPUT_DIM  = 5
HIDDEN_DIM = 64
K_HOPS     = 3
BATCH_SIZE = 32
GAMMA      = 0.99
EPS_START  = 1.0
EPS_END    = 0.05
EPS_DECAY  = 2000
TARGET_UPDATE = 50

print("Loading datasets...")
train_data = load_dataset('train')
test_data  = load_dataset('test')

print("Processing graph environments...")
train_envs = []
for data in train_data:
    if data['type'] == 'watts-strogatz':
        x        = extract_ba_features(data['graph']).to(device)
        norm_adj = normalized_adj(data['adj_matrix'])
        env      = MVCEnvironment(data['graph'])
        train_envs.append({'env': env, 'x': x, 'norm_adj': norm_adj})

print(f"Found {len(train_envs)} watts-strogatz training graphs.")

# Transfer neural network models to the selected device
policy_net = Structure2VecDQN(INPUT_DIM, HIDDEN_DIM, K_HOPS).to(device)
target_net = Structure2VecDQN(INPUT_DIM, HIDDEN_DIM, K_HOPS).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
memory = ReplayBuffer(10000)

epochs = 100
steps_done = 0
print(f"\nStarting DQN training for {epochs} epochs...")

for epoch in range(epochs):
    epoch_loss = 0.0
    random.shuffle(train_envs)

    for graph_data in train_envs:
        env      = graph_data['env']
        x        = graph_data['x']
        norm_adj = graph_data['norm_adj']

        state_mask = env.reset()
        done = False

        while not done:
            epsilon = EPS_END + (EPS_START - EPS_END) * np.exp(-1. * steps_done / EPS_DECAY)
            steps_done += 1

            with torch.no_grad():
                q_vals = policy_net(x, state_mask, norm_adj)

            action = select_action(q_vals, state_mask, epsilon)
            next_state_mask, reward, done = env.step(action)

            memory.push((x, norm_adj, state_mask.clone(), action, reward, next_state_mask.clone(), done))
            state_mask = next_state_mask

        # ==========================================
        # MOVED OUTSIDE THE WHILE LOOP
        # ==========================================
        # Optimize a few times at the end of the episode instead of every step
        if len(memory) >= BATCH_SIZE:
            # Do 3-5 batch updates per graph instead of 40+
            for _ in range(4):
                batch = memory.sample(BATCH_SIZE)
                loss = torch.tensor(0.0, device=device)

                for (b_x, b_adj, b_state, b_action, b_reward, b_next_state, b_done) in batch:
                    cur_q = policy_net(b_x, b_state, b_adj)[b_action]

                    if b_done:
                        target_q = torch.tensor(b_reward, dtype=torch.float32, device=device)
                    else:
                        with torch.no_grad():
                            next_q_vals = target_net(b_x, b_next_state, b_adj)
                            next_q_vals[b_next_state.squeeze(-1) == 1] = -float('inf')
                            max_next_q = next_q_vals.max()
                        target_q = b_reward + GAMMA * max_next_q

                    loss = loss + (cur_q - target_q) ** 2

                loss = loss / BATCH_SIZE
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
                optimizer.step()
                epoch_loss += loss.item()

        if steps_done % TARGET_UPDATE == 0:
            target_net.load_state_dict(policy_net.state_dict())
    print("EPOCH: ", epoch)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Avg Step Loss: {epoch_loss/max(1, steps_done):.4f} | Epsilon: {epsilon:.3f}")

# ==========================================
# 8. TESTING ON UNSEEN watts-strogatz GRAPHS
# ==========================================
print("\n" + "=" * 45)
print("TESTING ON UNSEEN watts-strogatz GRAPHS")
print("=" * 45)

policy_net.eval()

valid_count   = 0
better_count  = 0
equal_count   = 0
total         = 0
approx_ratios = []

for data in test_data:
    if data['type'] != 'watts-strogatz':
        continue

    x        = extract_ba_features(data['graph']).to(device)
    norm_adj = normalized_adj(data['adj_matrix'])
    env      = MVCEnvironment(data['graph'])

    state_mask = env.reset()
    done = False
    selected_nodes = []

    while not done:
        with torch.no_grad():
            q_vals = policy_net(x, state_mask, norm_adj)

        action = select_action(q_vals, state_mask, epsilon=0.0)
        state_mask, _, done = env.step(action)
        selected_nodes.append(action)

    predicted_size = len(selected_nodes)
    is_valid       = is_valid_vertex_cover(data['graph'], selected_nodes)

    approx_vc   = nx.algorithms.approximation.min_weighted_vertex_cover(data['graph'])
    approx_size = len(approx_vc)

    total += 1
    if is_valid:
        valid_count += 1
        ratio = predicted_size / approx_size
        approx_ratios.append(ratio)

    beats_baseline_strict = is_valid and (predicted_size < approx_size)
    matches_baseline      = is_valid and (predicted_size == approx_size)

    if beats_baseline_strict:
        better_count += 1
    if matches_baseline:
        equal_count += 1

    verdict = "beats baseline" if beats_baseline_strict else (
              "matches baseline" if matches_baseline else (
              "valid but larger" if is_valid else "INVALID COVER"))

    print(f"Graph ID : {data['id']} | Nodes: {data['num_nodes']}")
    print(f" > Valid Cover   : {is_valid}")
    print(f" > Predicted Size: {predicted_size}")
    print(f" > Baseline Size : {approx_size}  ({verdict})")
    print("-" * 40)

# ---- Final Report ----
print("\n" + "=" * 45)
print("FINAL RESULTS")
print("=" * 45)
validity     = (valid_count  / total * 100) if total > 0 else 0
beat_pct     = (better_count / total * 100) if total > 0 else 0
match_pct    = (equal_count  / total * 100) if total > 0 else 0
avg_ratio    = np.mean(approx_ratios) if approx_ratios else float('inf')

print(f"Total Graphs Tested   : {total}")
print(f"Valid Covers          : {valid_count}/{total}  ({validity:.1f}%)")
print(f"Beats Baseline        : {better_count}/{total}  ({beat_pct:.1f}%)")
print(f"Matches Baseline      : {equal_count}/{total}  ({match_pct:.1f}%)")
print(f"Avg Approx Ratio      : {avg_ratio:.3f}")
print("=" * 45)

Executing computations on device: cuda
Loading datasets...
Processing graph environments...
Found 1000 watts-strogatz training graphs.

Starting DQN training for 100 epochs...
EPOCH:  0
EPOCH:  1
EPOCH:  2
EPOCH:  3
EPOCH:  4
EPOCH:  5
EPOCH:  6
EPOCH:  7
EPOCH:  8
EPOCH:  9
Epoch 10/100 | Avg Step Loss: 0.0005 | Epsilon: 0.050
EPOCH:  10
EPOCH:  11
EPOCH:  12
EPOCH:  13
EPOCH:  14
EPOCH:  15
EPOCH:  16
EPOCH:  17
EPOCH:  18
EPOCH:  19
Epoch 20/100 | Avg Step Loss: 0.0001 | Epsilon: 0.050
EPOCH:  20
EPOCH:  21
EPOCH:  22
EPOCH:  23
EPOCH:  24
EPOCH:  25
EPOCH:  26
EPOCH:  27
EPOCH:  28
EPOCH:  29
Epoch 30/100 | Avg Step Loss: 0.0001 | Epsilon: 0.050
EPOCH:  30
EPOCH:  31
EPOCH:  32
EPOCH:  33
EPOCH:  34
EPOCH:  35
EPOCH:  36
EPOCH:  37
EPOCH:  38
EPOCH:  39
Epoch 40/100 | Avg Step Loss: 0.0001 | Epsilon: 0.050
EPOCH:  40
EPOCH:  41
EPOCH:  42
EPOCH:  43
EPOCH:  44
EPOCH:  45
EPOCH:  46
EPOCH:  47
EPOCH:  48
EPOCH:  49
Epoch 50/100 | Avg Step Loss: 0.0000 | Epsilon: 0.050
EPOCH:  50
EPO